## GitLab UI Plugin installieren

Das GitLab-Plugin wird als bestehendes Frontend- und Backend-Plugin installiert.

Die Backstage-App verwendet `app.packages: all`, deshalb werden Frontend-Plugins über Feature Discovery geladen.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd packages/app add @immobiliarelabs/backstage-plugin-gitlab
yarn --cwd packages/backend add @immobiliarelabs/backstage-plugin-gitlab-backend

In [ ]:
%%bash
cd ~/mybackstage/
mkdir -p packages/app/src/components/catalog/
cat <<EOF >packages/app/src/components/catalog/EntityPage.tsx
// packages/app/src/components/catalog/EntityPage.tsx

import {
  isGitlabAvailable,
  EntityGitlabContent,
  EntityGitlabLanguageCard,
  EntityGitlabMergeRequestsTable,
  EntityGitlabMergeRequestStatsCard,
  EntityGitlabPeopleCard,
  EntityGitlabPipelinesTable,
  EntityGitlabReadmeCard,
  EntityGitlabReleasesCard,
} from "@immobiliarelabs/backstage-plugin-gitlab";

//Farther down at the overviewContent declaration
//You can add only selected widgets or all of them.
const overviewContent = (
  <Grid container spacing={3} alignItems="stretch">
    <EntitySwitch>
      <EntitySwitch.Case if={isGitlabAvailable}>
        <Grid item md={12}>
          <EntityGitlabReadmeCard />
        </Grid>
        <Grid item sm={12} md={3} lg={3}>
          <EntityGitlabPeopleCard />
        </Grid>
        <Grid item sm={12} md={3} lg={3}>
          <EntityGitlabLanguageCard />
        </Grid>
        <Grid item sm={12} md={3} lg={3}>
          <EntityGitlabMergeRequestStatsCard />
        </Grid>
        <Grid item sm={12} md={3} lg={3}>
          <EntityGitlabReleasesCard />
        </Grid>
        <Grid item md={12}>
          <EntityGitlabPipelinesTable />
        </Grid>
        <Grid item md={12}>
          <EntityGitlabMergeRequestsTable />
        </Grid>
      </EntitySwitch.Case>
    </EntitySwitch>
  </Grid>
);
EOF

In [ ]:
%%bash
cd ~/mybackstage/
mkdir -p packages/backend/src/plugins
cat <<EOF >packages/backend/src/plugins/catalog.ts
// packages/backend/src/plugins/catalog.ts
import { GitlabFillerProcessor } from "@immobiliarelabs/backstage-plugin-gitlab-backend";

export default async function createPlugin(
  env: PluginEnvironment,
): Promise<Router> {
  const builder = await CatalogBuilder.create(env);
  //...
  // Add this line
  builder.addProcessor(new GitlabFillerProcessor(env.config));
  //...
  const { processingEngine, router } = await builder.build();
  await processingEngine.start();
  return router;
}
EOF


In [ ]:
%%bash
cd ~/mybackstage/
mkdir -p packages/backend/src/plugins
cat <<EOF >packages/backend/src/plugins/gitlab.ts
// packages/backend/src/plugins/gitlab.ts
import { PluginEnvironment } from "../types";
import { Router } from "express-serve-static-core";
import { createRouter } from "@immobiliarelabs/backstage-plugin-gitlab-backend";

export default async function createPlugin(
  env: PluginEnvironment,
): Promise<Router> {
  return createRouter({
    logger: env.logger,
    config: env.config,
  });
}
EOF



In [ ]:
%%bash
cd ~/mybackstage/
mkdir -p packages/backend/src/plugins


Das Backend-Plugin wird im neuen Backend-System registriert.

In [ ]:
%%bash
cd ~/mybackstage/
grep -q "@immobiliarelabs/backstage-plugin-gitlab-backend" packages/backend/src/index.ts || sed -i "/backend.start();/i backend.add(import('@immobiliarelabs/backstage-plugin-gitlab-backend'));"   packages/backend/src/index.ts


GitLab wird über die normale Backstage-Integration konfiguriert.

In [ ]:
%%bash
cd ~/mybackstage/

cat > app-config.local.yaml <<'EOF'
integrations:
  gitlab:
    - host: gitlab.com
      # token: ${GITLAB_TOKEN}

# app-config.yaml
# ...
gitlab:
  # Default path for CODEOWNERS file
  # Default: CODEOWNERS
  defaultCodeOwnersPath: .gitlab/CODEOWNERS
  # Default path for README file
  # Default: README.md
  defaultReadmePath: .gitlab/README.md
  # Entity Kinds to which the plugin works, if you want to render gitlab
  # information for one Kind you have to add it in this list.
  # Default: ['Component']
  allowedKinds: ["Component", "Resource"]
  # This parameter controls SSL Certs verification
  # Default: true
  proxySecure: false
  # Activate Oauth/OIDC
  # Default: false
  useOAuth: false
  # Cache configuration
  cache:
    # Enable caching for the Gitlab plugin
    # Default: false
    enabled: true
    # Cache TTL for the Gitlab plugin in seconds
    # Default: 300
    ttl: 300
EOF


Eine Catalog-Component benötigt die GitLab-Projektannotation.

    metadata:
      annotations:
        gitlab.com/project-slug: gruppe/projekt

**Backstage starten:** Backstage wird mit dem neuen Plugin gestartet.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Produktion"
export BACKSTAGE_PORT="3002"

echo "http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn start --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.production.yaml --config ~/mybackstage/app-config.local.yaml